# Load User Attributes into Neo4j

Load the two final node labels (`:Occupation`, `:Gender`) and the two relationship types that connect users to them (`:HAS_OCCUPATION`, `:HAS_GENDER`). This completes the user side of the graph and lets us run user-attribute-based collaborative queries.

Requires [`04_load_nodes.ipynb`](04_load_nodes.ipynb) to have run — we need the 943 `:User` nodes as edge endpoints.

## Scope

| Element | Source | Expected count |
|---|---|---|
| `:Occupation {name}` | `data/raw/ml-100k/u.occupation` | 21 |
| `:Gender {name}` | hardcoded `['M', 'F']` | 2 |
| `(:User)-[:HAS_OCCUPATION]->(:Occupation)` | `u.user.occupation` | 943 |
| `(:User)-[:HAS_GENDER]->(:Gender)` | `u.user.gender` | 943 |

## Not in scope

- Derived edges between users (e.g. `:SAME_OCCUPATION_AS`) — these are projections over the data, not source data, and land later.

## The pattern (recap)

Same as 04 (nodes) and 05 (edges):
1. **Python** reads the source file and returns either unique names (for nodes) or edge rows (for relationships).
2. **Cypher** `MATCH`es both endpoints by their unique key and `MERGE`s the edge.

## Setup

In [1]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.etl import (
    read_occupations, read_genders,
    read_user_occupation, read_user_gender,
    to_records,
)
from src.graph import run_query, run_write_query, close_driver

## Optional teardown — surgical wipe

Wipes only `:Occupation` + `:Gender` nodes and their `:HAS_OCCUPATION` / `:HAS_GENDER` edges. Everything else (Users, Movies, Genres, Directors, Actors and the four edge types from `05_load_relationships.ipynb`) is untouched.

Unlike the blanket teardowns in 04 and 05, this one is **label-scoped** — so it's safe to run, but you only need it once: the load itself is idempotent via `MERGE`. Uncomment if you have leftover state from a prior experiment, then re-comment.

In [2]:
# run_write_query("MATCH ()-[r:HAS_OCCUPATION|HAS_GENDER]->() DELETE r")
# run_write_query("MATCH (n:Occupation) DETACH DELETE n")
# run_write_query("MATCH (n:Gender)     DETACH DELETE n")
# print("Cleaned :Occupation, :Gender, :HAS_OCCUPATION, :HAS_GENDER.")

## Schema artifacts — uniqueness constraints

Same pattern as 04: a uniqueness constraint creates a backing index automatically, which is what lets the `MATCH (o:Occupation {name: ...})` lookups in the edge load run as O(1) index hits instead of label scans.

In [3]:
run_write_query("CREATE CONSTRAINT occupation_name IF NOT EXISTS FOR (o:Occupation) REQUIRE o.name IS UNIQUE")
run_write_query("CREATE CONSTRAINT gender_name     IF NOT EXISTS FOR (g:Gender)     REQUIRE g.name IS UNIQUE")
print("Constraints in place.")

Constraints in place.


## 1 — :Occupation nodes

21 nodes, one per occupation in `u.occupation` (file order). Same source-of-truth pattern as `:Genre` ← `u.genre`.

In [4]:
occupations = read_occupations()
print(f"Read {len(occupations)} occupations.")
print(f"  sample: {occupations[:5]}")

Read 21 occupations.
  sample: ['administrator', 'artist', 'doctor', 'educator', 'engineer']


In [5]:
cypher = """
UNWIND $rows AS row
MERGE (:Occupation {name: row.name})
"""
run_write_query(cypher, {"rows": [{"name": o} for o in occupations]})

count = run_query("MATCH (o:Occupation) RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count} :Occupation nodes.")

Neo4j now has 21 :Occupation nodes.


## 2 — :Gender nodes

Just 2 nodes: `M` and `F`. Hardcoded since the source has only those two values and we kept them raw (no `M`→`Male` normalization). The two nodes will each connect to ~470 users, so expect a super-hub shape — fine for our query patterns.

In [6]:
genders = read_genders()
print(f"Read {len(genders)} genders: {genders}")

Read 2 genders: ['M', 'F']


In [7]:
cypher = """
UNWIND $rows AS row
MERGE (:Gender {name: row.name})
"""
run_write_query(cypher, {"rows": [{"name": g} for g in genders]})

count = run_query("MATCH (g:Gender) RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count} :Gender nodes.")

Neo4j now has 2 :Gender nodes.


## 3 — HAS_OCCUPATION edges

943 edges, one per user. The reader re-reads `u.user` independently rather than widening `read_users()`'s contract. Property-less edge — same shape as `IN_GENRE`.

In [8]:
user_occ_df = read_user_occupation()
print(f"Read {len(user_occ_df):,} user-occupation edge rows from u.user.")
user_occ_df.head()

Read 943 user-occupation edge rows from u.user.


,userId,occupation
0,1,technician
1,2,other
2,3,writer
3,4,technician
4,5,other


In [9]:
cypher = """
UNWIND $rows AS row
MATCH (u:User       {userId: row.userId})
MATCH (o:Occupation {name:   row.occupation})
MERGE (u)-[:HAS_OCCUPATION]->(o)
"""
run_write_query(cypher, {"rows": to_records(user_occ_df)})

count = run_query("MATCH ()-[:HAS_OCCUPATION]->() RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count:,} :HAS_OCCUPATION edges.")

Neo4j now has 943 :HAS_OCCUPATION edges.


## 4 — HAS_GENDER edges

943 edges, one per user. Identical shape to HAS_OCCUPATION — only the labels and column name change.

In [10]:
user_gen_df = read_user_gender()
print(f"Read {len(user_gen_df):,} user-gender edge rows from u.user.")
user_gen_df.head()

Read 943 user-gender edge rows from u.user.


,userId,gender
0,1,M
1,2,F
2,3,M
3,4,M
4,5,F


In [11]:
cypher = """
UNWIND $rows AS row
MATCH (u:User   {userId: row.userId})
MATCH (g:Gender {name:   row.gender})
MERGE (u)-[:HAS_GENDER]->(g)
"""
run_write_query(cypher, {"rows": to_records(user_gen_df)})

count = run_query("MATCH ()-[:HAS_GENDER]->() RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count:,} :HAS_GENDER edges.")

Neo4j now has 943 :HAS_GENDER edges.


## Summary — edge counts by relationship type

Full tally across every relationship type. After this notebook runs cleanly you should see 6 types (the 4 from `05_load_relationships.ipynb` + the 2 new). Ghost counts from prior experiments should be gone, assuming the teardown was uncommented for the first run.

In [12]:
result = run_query("""
MATCH ()-[r]->()
RETURN type(r) AS relationship, count(*) AS n
ORDER BY n DESC
""")
print(f"{'relationship':<20} {'count':>10}")
print("-" * 32)
for row in result:
    print(f"{row['relationship']:<20} {row['n']:>10,}")

relationship              count
--------------------------------
RATED                   100,000
FEATURES_ACTOR           16,243
IN_GENRE                  2,893
DIRECTED_BY               1,790
HAS_OCCUPATION              943
HAS_GENDER                  943


## Spot-check 1 — User 196's full profile

Pulls user 196's age, occupation, and gender in one query. `OPTIONAL MATCH` makes it robust if any attribute is somehow missing (it shouldn't be, but the pattern is defensive).

In [13]:
result = run_query("""
MATCH (u:User {userId: 196})
OPTIONAL MATCH (u)-[:HAS_OCCUPATION]->(o:Occupation)
OPTIONAL MATCH (u)-[:HAS_GENDER]->(g:Gender)
RETURN u.userId AS userId, u.age AS age, o.name AS occupation, g.name AS gender
""")
row = result[0]
print(f"User ID:    {row['userId']}")
print(f"Age:        {row['age']}")
print(f"Occupation: {row['occupation']}")
print(f"Gender:     {row['gender']}")

User ID:    196
Age:        49
Occupation: writer
Gender:     M


## Spot-check 2 — Genres liked by users sharing User 196's occupation

Three hops in one Cypher block: `User → HAS_OCCUPATION → Occupation ← HAS_OCCUPATION ← peers → RATED (4★+) → Movie → IN_GENRE → Genre`.

This is the payoff query: it shows that the user-attribute edges unlock a *collaborative-filtering-flavored* query — "what do people like me enjoy?" — in plain Cypher, with no aggregation tables, no joins, just a traversal.

In [14]:
result = run_query("""
MATCH (u:User {userId: 196})-[:HAS_OCCUPATION]->(o:Occupation)<-[:HAS_OCCUPATION]-(peer:User)
MATCH (peer)-[r:RATED]->(m:Movie)-[:IN_GENRE]->(g:Genre)
WHERE r.rating >= 4
RETURN g.name AS genre, count(*) AS high_ratings
ORDER BY high_ratings DESC LIMIT 5
""")
print(f"{'genre':<20} {'high_ratings':>12}")
print("-" * 34)
for row in result:
    print(f"{row['genre']:<20} {row['high_ratings']:>12}")

genre                high_ratings
----------------------------------
Drama                        1202
Comedy                        674
Thriller                      661
Action                        616
Romance                       516


## Idempotency check (re-run after the cells above)

Re-run the summary cell. Counts should be **identical** — no double-loads. Same `MERGE` guarantee as in 04 and 05.

In [15]:
close_driver()
print("Driver closed.")

Driver closed.
